# Stock forecast

A ML classification model for predicting stock market trends.

## Getting Data


In [ ]:
import yfinance as yf

# obtain the company’s stock data for the last year:
tkr = yf.Ticker('F')
hist = tkr.history(period="3y")
hist.index = hist.index.tz_localize(None)
print(hist)

In [ ]:
import pandas_datareader.data as pdr
from datetime import date, timedelta

# obtain one year’s worth of S&P 500 data from the Stooq website
end = date.today()
start = end - timedelta(days=365)  # calculates the exact date one year ago
index_data = pdr.get_data_stooq('^SPX', start, end)
index_data = index_data.sort_values('Date', ascending=True)  # reorder the records on ascendent order
print(index_data)

In [ ]:
import pandas as pd

# Combine the data into a single DataFrame
hist, index_data = hist.reset_index(), index_data.reset_index()

#print(hist)
#print(index_data)


# Merge on 'Date' with suffixes
df = pd.merge(
    hist, 
    index_data, 
    on='Date', 
    suffixes=('', '_idx'),   # Adjust suffixes as needed
    how='outer'                      # Use 'outer' to keep all dates
)

df = df.set_index("Date")

print(df)

In [ ]:
# We only want to use the closing prices and volumes
df = df[['Close','Volume','Close_idx','Volume_idx']].rename(columns = {"Close": "Price", "Close_idx": "Price_idx"})
print(df)

## Deriving Features from Continuous Data


In [ ]:
import sys

sys.path.append(r'..\scripts')


import ts_analysis_techn as ts


variablesToCalcullatePercentageChanges = ["Price", "Volume", "Price_idx", "Volume_idx"]

# Getting percentage changes from our defined list of columns
engineer = ts.TimeSeries(df, variablesToCalcullatePercentageChanges)
engineer.add_percentage_change()
df = df.dropna()


print(df)
print("\n")
print(df.columns)


In [ ]:
# We only want the percentage changes to become features in the machine learning model
feature_columns = df[['1dayPriceRise','1dayVolumeRise','1dayPriceRise_idx','1dayVolumeRise_idx']]

print(feature_columns)

## Generating the Output Variable


The algorithm implemented here assumes the following:
1. A price increase of more than 1 percent in relation to the next day is 
regarded as a rise (1).
2. A price decrease by more than 1 percent in relation to the next day is 
regarded as a fall (-1).
3. The rest is regarded as stagnation (0).

In [ ]:
import numpy as np

conditions = [
 (df['PriceRise'].shift(-1) > 0.01),
 (df['PriceRise'].shift(-1)< -0.01)
]

choices = [1, -1]
df['Pred'] = np.select(conditions, choices, default=0)


## Training and Evaluating the Model

In [ ]:
features = feature_columns.to_numpy()
features = np.around(features, decimals=2)
target = df['Pred'].to_numpy()


In [ ]:
from sklearn.model_selection import train_test_split
rows_train, rows_test, y_train, y_test = train_test_split(features, target, test_size=0.2)
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
clf.fit(rows_train, y_train)

In [ ]:
print(clf.score(rows_test, y_test))